# syncsan in Colab

This notebook installs [syncsan](https://github.com/ShreshthRajan/syncsan), builds it for the notebook's GPU, and runs it on a 30-line kernel with a race in it. Pick a GPU runtime first (Runtime, then Change runtime type, then T4 GPU).

In [ ]:
%%bash
nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader
pip -q install syncsan
syncsan build --arch $(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .) > build.log 2>&1 && echo built || tail -20 build.log

The kernel in `examples/handoff.cu`: one block writes data, fences and bumps a counter, and another block spins on the counter and reads the data with no fence after the spin. It gives the right answer on the GPU. syncsan reports the race, and adversarial mode shows the wrong answer the PTX memory model allows.

In [ ]:
%%bash
[ -d syncsan ] || git clone -q https://github.com/ShreshthRajan/syncsan
cd syncsan
ARCH=sm_$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
nvcc -arch=$ARCH -lineinfo -o handoff examples/handoff.cu
echo '--- native'; ./handoff
echo '--- syncsan'; syncsan run -- ./handoff 2>&1
echo '--- adversarial'; SYNCSAN_ADVERSARIAL=1 syncsan run -- ./handoff 2>&1 || true

Add the missing fence with `-DFIXED` and syncsan reports nothing.

In [ ]:
%%bash
cd syncsan
ARCH=sm_$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
nvcc -arch=$ARCH -lineinfo -DFIXED -o handoff_fixed examples/handoff.cu
syncsan run -- ./handoff_fixed 2>&1; SYNCSAN_ADVERSARIAL=1 syncsan run -- ./handoff_fixed 2>&1

Optional: the 51 known-answer tests (a few minutes).

In [ ]:
%%bash
cd syncsan
ARCH=sm_$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
bash tests/run_tests.sh $ARCH > tests.log 2>&1
echo "PASS $(grep -c ^PASS tests.log)  FAIL $(grep -c ^FAIL tests.log)  SKIP $(grep -c ^SKIP tests.log)"; grep -q ^PASS tests.log || tail -20 tests.log; grep -A8 ^FAIL tests.log | head -60